<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Choose a recogniser for two languages

**[Choose a recogniser for two languages](https://learning.nextia-ai.com/courses/speech/m10/choose-a-recogniser-for-two-languages/)** · Speech AI: Transcribe, Understand and Speak · Case study 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** compare Vosk small and three Whisper sizes on real English and French speech from FLEURS, by language and by speaker gender, with the course's normalisation guide. You will put the error next to the speed, memory and cost, choose a recogniser for each language, and run your choice on new recordings.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page; the notebook runs in about 3 minutes |
| **Needs** | An internet connection for the setup cells (about 30 MB of files and about 480 MB for the Whisper model). No account, no key and no graphics card. |
| **You should know** | WER, the normalisation guide and slices, from [Word error rate, step by step](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/), [Normalise before you compare](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/) and [Whisper sizes on a laptop](https://learning.nextia-ai.com/courses/speech/m04/whisper-sizes-on-a-laptop/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, soundfile 0.14.0, matplotlib 3.11.2 and faster-whisper 1.2.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m10/choose-a-recogniser-for-two-languages/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M10-L01-choose-a-recogniser-for-two-languages/choose-a-recogniser-for-two-languages-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, soundfile 0.14.0 and faster-whisper 1.2.1, and matplotlib for the pictures. It takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ["numpy", "soundfile", "faster-whisper"]:
    print(name, version(name))

## Setup: get the recordings and the transcripts

Run the next cell. It downloads the files into a folder `fleurs-recognisers` and checks each file's checksum:

- from FLEURS on Hugging Face (the original source, at a fixed version): the four files with the sentence text of each recording (`test.tsv` and `dev.tsv`, English and French);
- from the Nextia labs repository: the list of recordings in our sample, the recorded transcripts of five recognisers, their speeds, 23 short recordings (FLAC), and the course's helper `compare.py`.

It needs no account. You should see `Ready:` and the number of files. If you see `Could not download`, check your connection and run the cell again; if you see `wrong checksum`, delete the folder `fleurs-recognisers` and run the cell again.

**About the data.** FLEURS (Conneau et al., 2022, Google) has volunteers who read Wikipedia sentences aloud in 102 languages. It is licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Our changes: we kept a sample of the test split (and 20 recordings of the validation split) and stored the clips as 16-bit FLAC. The [dataset card](dataset.md) has the details.

In [ ]:
# Setup: download the files into fleurs-recognisers/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
    "data/audio/en-1870-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-1870-f.flac",
        "8ba7f365d51651a7513cfbe59b1505b07b0733f054ef4f225aebadc44c04d6e4"
    ],
    "data/audio/en-1870-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-1870-m.flac",
        "d1dd770e1cd2df5489aab23049c341991c538b12d143e3ad3902ae1bd48aeeab"
    ],
    "data/audio/en-new01-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new01-f.flac",
        "339520ccd0f575d7650de7b16deb7f10f7ea2c9d38e2c0cc728f3145b4ba92be"
    ],
    "data/audio/en-new02-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new02-f.flac",
        "98f371aff22fdf2df569361fee78646e8b49e2c02faa9f6df38f7d11b763e9ed"
    ],
    "data/audio/en-new03-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new03-f.flac",
        "029735a544834185c0ece4a16a5551ef63925602401af101b1827346b537c16f"
    ],
    "data/audio/en-new04-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new04-f.flac",
        "8ba2cf7f42beb6830a7cbe1bb8a841ba52fbbaedd57adf678084bfd6e1958b1a"
    ],
    "data/audio/en-new05-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new05-f.flac",
        "99eef2e2805bca443a285ce92ea75eb624d32692acdfa3a69e8fb00dc4d9f212"
    ],
    "data/audio/en-new06-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new06-m.flac",
        "e2513d4232fcd8197bf14529e4e5ba3c0a6c56cda6f74fe390f6b1e7f5bfe012"
    ],
    "data/audio/en-new07-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new07-m.flac",
        "037aeaf54b281b5793616a9bc5787d538021a76c2a56eb950fd20664f4ca1d02"
    ],
    "data/audio/en-new08-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new08-m.flac",
        "74fd4e27b379eb1e2703033d38c2a504b6db7feb353e4ccc16f6707651aaff1a"
    ],
    "data/audio/en-new09-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new09-m.flac",
        "b99671ae3ddcedf54d5ba6f764900f0c35f47143a15965d19ab7d29b8cddbf7f"
    ],
    "data/audio/en-new10-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/en-new10-m.flac",
        "3c3a351a97c28225859958b05d1931093eae2f3b44b041a01a4e05e969339f7f"
    ],
    "data/audio/fr-1870-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-1870-m.flac",
        "a758f51872ce7c420617ebc3b6f46237c947fc1168e3ef1e2e589e6124bf7747"
    ],
    "data/audio/fr-new01-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new01-f.flac",
        "8cb662ac8df8ab79029743b1f7e9999e845d9c795bd712fb2a5442b6c0cbf558"
    ],
    "data/audio/fr-new02-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new02-f.flac",
        "ca801ffa434a6954145c5d581e676a995debfb63da6b88ad0c703a2730a99f77"
    ],
    "data/audio/fr-new03-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new03-f.flac",
        "0d6723535eacba7f7839831114d3b39f9c6a9639653d5535987da56f15638fac"
    ],
    "data/audio/fr-new04-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new04-f.flac",
        "fdeae92eb57e179734d3e8763abaeb855bb5ad182071e096ae4fb7c8d4a66bb9"
    ],
    "data/audio/fr-new05-f.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new05-f.flac",
        "614f498dd71da35aaa42f006db60b0f679bb8ba306336edbcf47dd8e90470586"
    ],
    "data/audio/fr-new06-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new06-m.flac",
        "6e280821cf3f56745bc9e55e6b93b6f074416cb44cc5ee316194e12059bdab15"
    ],
    "data/audio/fr-new07-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new07-m.flac",
        "ed0780b2c0b11710993c7608b87c5a3664b77e38698369b09841a2678eba4194"
    ],
    "data/audio/fr-new08-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new08-m.flac",
        "902b0184bc55906fef5ec6d2fb5513218866e034406581a559ffc720ac209372"
    ],
    "data/audio/fr-new09-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new09-m.flac",
        "bf39d50a0991e62e38bcc267cbab7b22461b88e451ab3066a751b35822e3f507"
    ],
    "data/audio/fr-new10-m.flac": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/audio/fr-new10-m.flac",
        "b28e75c45cf4ac2aa7051ecff1b243e99df728d6a853f8509b7f23dae4de4f9e"
    ],
    "data/recordings.json": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/recordings.json",
        "2e5e5ba22ab45483d0bb90b3c7a60358e3417a4a434e26341c5f4889e5931635"
    ],
    "data/runs.json": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/runs.json",
        "4c6d4e3ea5b702257b31cf67bafa757d657ddcf3314550517c53fd292a1e76bf"
    ],
    "data/transcripts/vosk-small-en-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/vosk-small-en-new.jsonl",
        "d3326e006c56f556f5a0731c5172a4043168971b8319914506154b3634cb2a4b"
    ],
    "data/transcripts/vosk-small-en.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/vosk-small-en.jsonl",
        "8d96e6ec705419efb43cc470bd97457265e8d256a16b320dd31904b8a6c5237a"
    ],
    "data/transcripts/vosk-small-fr-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/vosk-small-fr-new.jsonl",
        "d41aa13cd4061dd8f7d181c924be76aaa9776a099dc4c4f8a5c907667c2b32ac"
    ],
    "data/transcripts/vosk-small-fr.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/vosk-small-fr.jsonl",
        "436785d8b6c6f5171ff633ba1a2c92e914ddb71c49ed7ed81f021d007cda14ac"
    ],
    "data/transcripts/whisper-base-en-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-base-en-new.jsonl",
        "a696f655854f725a30490aa2f61750524834d45839e2b1002c3d9a475b65e032"
    ],
    "data/transcripts/whisper-base-en.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-base-en.jsonl",
        "34d87d864c5e5218768151c08074d5fe4433778cf0839ca996c9a801c1c814bf"
    ],
    "data/transcripts/whisper-base-fr-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-base-fr-new.jsonl",
        "2c14d37cf50b520c896640f8be8fd91e52d3f831daf1cac0d4253003bf8d374d"
    ],
    "data/transcripts/whisper-base-fr.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-base-fr.jsonl",
        "a522e3a01c68ed4788150e4d1bdcc6ea325464ff0496d088de11d1c24f0d0a0c"
    ],
    "data/transcripts/whisper-medium-en.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-medium-en.jsonl",
        "3a2cad46eaab1a0870f7478814bd5079e0e25345eabdcdfeb70f6eff5e10dba8"
    ],
    "data/transcripts/whisper-medium-fr-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-medium-fr-new.jsonl",
        "ed2f2849f079e66d46165ad0b5d5b1e58749ff4da487adb50cf9e8d0585ef46a"
    ],
    "data/transcripts/whisper-medium-fr.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-medium-fr.jsonl",
        "e1721f05d94528490981a1b28b783b6e534e9d1c7ed388adaca87764f1d85b78"
    ],
    "data/transcripts/whisper-small-auto-en.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-small-auto-en.jsonl",
        "836a6ccd25dd3db791e9dacf6bfccc2e00571ec47d686089a32ff84988cb9cbd"
    ],
    "data/transcripts/whisper-small-auto-fr.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-small-auto-fr.jsonl",
        "089974980add51f172af61e27402c6256e20a637c729ded74c0e435ba38b4aa8"
    ],
    "data/transcripts/whisper-small-en-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-small-en-new.jsonl",
        "0bf27e0a1eb2139c835e57fcc3777fcb8bd31d7f248386c349f4edfa1e175c58"
    ],
    "data/transcripts/whisper-small-en.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-small-en.jsonl",
        "4410d4b6d8af1543d75e283349bbc047222d226ea4f5b4325ef481c0ddb578c0"
    ],
    "data/transcripts/whisper-small-fr-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-small-fr-new.jsonl",
        "3cdcc315bd2a8fbe1fd8fd93b045e9be8c650354417c62759d1d4729fc4a1c5d"
    ],
    "data/transcripts/whisper-small-fr.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-small-fr.jsonl",
        "0f1f12e86769b9cb3a6fdb872af6c3007107ce617586f779f9a3511a2ed99bb7"
    ],
    "data/transcripts/whisper-tiny-en-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-tiny-en-new.jsonl",
        "c705ab589acd892074d5c72b3f3b4095a235ffcb2392a79c566949204b4eddae"
    ],
    "data/transcripts/whisper-tiny-en.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-tiny-en.jsonl",
        "d8606ba5b1308422d5838aab91bb823b12c31347b253b62f4e743ea8ac2acd8e"
    ],
    "data/transcripts/whisper-tiny-fr-new.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-tiny-fr-new.jsonl",
        "26dfd91245d2c5d171449317c0bcfae389fe645ae2f58b015094467d19bb43e7"
    ],
    "data/transcripts/whisper-tiny-fr.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/M10-L01-choose-a-recogniser-for-two-languages/data/transcripts/whisper-tiny-fr.jsonl",
        "cc3df0f7cc071f22a66fb02f3bec7c2571c305b7e7c768328020d1e67957e6f5"
    ],
    "fleurs/en_us-test.tsv": [
        "https://huggingface.co/datasets/google/fleurs/resolve/70bb2e84b976b7e960aa89f1c648e09c59f894dd/data/en_us/test.tsv",
        "74c046239374deeb60fa63f258f907388093a32bcaa3140965f70ef05c79f7ca"
    ],
    "fleurs/fr_fr-test.tsv": [
        "https://huggingface.co/datasets/google/fleurs/resolve/70bb2e84b976b7e960aa89f1c648e09c59f894dd/data/fr_fr/test.tsv",
        "5d06d338b242e00786fcf12c4c92008b9f399d5a5c872c91dca90572e7869c0d"
    ],
    "fleurs/en_us-dev.tsv": [
        "https://huggingface.co/datasets/google/fleurs/resolve/70bb2e84b976b7e960aa89f1c648e09c59f894dd/data/en_us/dev.tsv",
        "9d57ee7e91e9d4c92edb39f6bbea668ef8dc2a3ff96eb510d5580b2ad05d17ec"
    ],
    "fleurs/fr_fr-dev.tsv": [
        "https://huggingface.co/datasets/google/fleurs/resolve/70bb2e84b976b7e960aa89f1c648e09c59f894dd/data/fr_fr/dev.tsv",
        "3e0b792358c1cb4a426fe1c18fc1571d5406b390d738a2ed1bfd3c8b9d28de44"
    ],
    "compare.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/compare.py",
        "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b"
    ]
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("fleurs-recognisers").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(url, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder fleurs-recognisers and run again.")

os.chdir(PROJECT)
print(f"Ready: {len(FILES)} files in fleurs-recognisers.")

## Setup: helpers

The next cell defines the helpers. Read their one-line descriptions:

- `recordings` and `new_recordings`: each recording's language, sentence, speaker gender, length and text.
- `transcripts(method, language)`: the recorded transcripts of one recogniser, with the compute time.
- `norm(text, language, guide)`: the normalisation guide. `guide="full"` is the course's guide; `"full+"` adds one rule for big numbers (section 2).
- `errors(ref, hyp, language)` and `score(method, language, gender)`: word errors, WER, CER and the real-time factor.
- `expect` and `expect_hash`: the checks after each task.

Run the cell. You should see the number of recordings.

In [ ]:
# Helpers: recordings, transcripts, the guide, scores, and checks that do not show the answer.
import csv, hashlib, json, random, warnings
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def read_tsv(path):
    """FLEURS's own file: the sentence text of each recording, by file name."""
    rows = csv.reader(open(path, encoding="utf-8"), delimiter="\t", quoting=csv.QUOTE_NONE)
    return {r[1]: r[2] for r in rows}

info = json.load(open("data/recordings.json"))
TEXT = {}
for code_, lang in (("en_us", "en"), ("fr_fr", "fr")):
    for split in ("test", "dev"):
        TEXT.update(read_tsv(f"fleurs/{code_}-{split}.tsv"))
recordings = {r["utt_id"]: {**r, "text": TEXT[r["file"]]} for r in info["sample"]}
new_recordings = {r["utt_id"]: {**r, "text": TEXT[r["file"]]} for r in info["new"]}
runs = {r["name"]: r for r in json.load(open("data/runs.json"))}

def transcripts(method, language, new=False):
    """The recorded transcripts of one recogniser: {utt_id: {"text": ..., "seconds": compute time}}."""
    path = f"data/transcripts/{method}-{language}{'-new' if new else ''}.jsonl"
    return {r["utt_id"]: r for r in map(json.loads, open(path, encoding="utf-8")) if r}

BIG = {"en": {"hundred": 100, "hundreds": 100, "thousand": 1000, "thousands": 1000, "million": 10**6, "millions": 10**6},
       "fr": {"cent": 100, "cents": 100, "mille": 1000, "million": 10**6, "millions": 10**6}}
ONE = {"en": ("a", "one"), "fr": ("un",)}

def big_numbers(text, language):
    """Our addition to the guide: "2 thousand and 5" -> "2005", "a hundred" -> "100", "40 mille" -> "40000"."""
    tokens, out, i = text.split(), [], 0
    big = BIG[language]
    while i < len(tokens):
        j, total, current, seen = i, 0, 0, False
        while j < len(tokens):
            t = tokens[j]
            nxt = tokens[j + 1] if j + 1 < len(tokens) else ""
            if t.isdigit() and (current == 0 or (current % 100 == 0 and int(t) < 100)):
                current += int(t)
            elif t in big and (current or seen or language == "fr" or (j > i and tokens[j - 1] in ONE[language])):
                if big[t] == 100:
                    current = max(current, 1) * 100
                else:
                    total, current = total + max(current, 1) * big[t], 0
                seen = True
            elif t in ONE[language] and nxt in big and j == i:
                current = 1
            elif t == "and" and seen and current % 100 == 0 and nxt.isdigit() and int(nxt) < 100:
                pass
            else:
                break
            j += 1
        if seen:
            out.append(str(total + current))
            i = j
        else:
            out.append(tokens[i])
            i += 1
    return " ".join(out)

def norm(text, language, guide="full+"):
    """guide: "none", "basic", "full" (the course's guide) or "full+" (the guide and big_numbers)."""
    if guide == "full+":
        return big_numbers(cp.normalise(text, "full", language), language)
    return cp.normalise(text, guide, language)

def errors(ref, hyp, language, guide="full+"):
    """(word errors, reference words) of one recording."""
    c = cp.counts(norm(ref, language, guide), norm(hyp, language, guide))
    return c["sub"] + c["del"] + c["ins"], c["ref_words"]

def score(method, language, gender=None, guide="full+", rows=None, new=False):
    """WER and CER in %, errors summed over the recordings, and the real-time factor."""
    rows = rows or (new_recordings if new else recordings)
    rec = transcripts(method, language, new)
    ids = [u for u, r in rows.items() if r["language"] == language and gender in (None, r["gender"]) and u in rec]
    e = n = ce = cn = 0
    for u in ids:
        r, h = norm(rows[u]["text"], language, guide), norm(rec[u]["text"], language, guide)
        c = cp.counts(r, h)
        e, n = e + c["sub"] + c["del"] + c["ins"], n + c["ref_words"]
        ce, cn = ce + cp.distance(r, h), cn + len(r)
    rtf = sum(rec[u]["seconds"] for u in ids) / sum(rows[u]["seconds"] for u in ids)
    return {"recordings": len(ids), "words": n, "wer": round(100 * e / n, 1), "cer": round(100 * ce / cn, 1),
            "rtf": round(rtf, 3)}

def fingerprint(value):
    def norm_(v):
        if hasattr(v, "tolist"):
            return norm_(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm_(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm_(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm_(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm_(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

METHODS = ["vosk-small", "whisper-tiny", "whisper-base", "whisper-small"]
print(len(recordings), "recordings in the sample,", len(new_recordings), "new recordings")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data, briefly

Inès, the analyst, checks what she has before she scores anything: how many recordings in each language and gender, and whether every sentence was read twice, once by a woman and once by a man.

In [ ]:
from collections import Counter
print(Counter((r["language"], r["gender"]) for r in recordings.values()))
for lang in ("en", "fr"):
    rows = [r for r in recordings.values() if r["language"] == lang]
    sentences = Counter(r["sentence_id"] for r in rows)
    print(lang, len(rows), "recordings,", len(sentences), "sentences,",
          "every sentence read twice:", set(sentences.values()) == {2},
          f"{sum(r['seconds'] for r in rows) / 60:.1f} minutes")

> **Check.** 262 English and 230 French recordings, half female and half male. Every sentence was read twice. So when we compare women and men, we compare the same sentences.

Listen to one sentence, read by a woman and by a man in English, and by a man in French. The players never start by themselves.

In [ ]:
for uid in ['en-1870-f', 'en-1870-m', 'fr-1870-m']:
    r = recordings[uid]
    print(f"{uid} ({r['gender']}, {r['seconds']} s): {r['text']}")
    display(Audio(f"data/audio/{uid}.flac"))

## 2. Check the guide on new text

The course's normalisation guide was written for Larkfield's calls, where order numbers are said digit by digit. FLEURS sentences have years and big numbers. Vosk writes numbers as words ("two thousand and five"), Whisper mostly as digits.

> **Predict.** If the guide does not turn "two thousand and five" into "2005", which recogniser is punished more: Vosk or Whisper?

In [ ]:
said = recordings["en-1701-m"]["text"]
vosk_wrote = transcripts("vosk-small", "en")["en-1701-m"]["text"]
print("said:           ", said)
print("Vosk wrote:     ", vosk_wrote)
print("course's guide: ", norm(said, "en", "full"), "|", norm(vosk_wrote, "en", "full"))
print("+ big_numbers:  ", norm(said, "en"), "|", norm(vosk_wrote, "en"))
for m in METHODS:
    print(f"{m:14}", {lang: (score(m, lang, guide="full")["wer"], score(m, lang)["wer"]) for lang in ("en", "fr")})

> **Check.** With the course's guide, Vosk has 25.2% in English; with the added rule, @VOSK_EN@%. Whisper barely moves. A guide that does not fit the data punishes one recogniser more than another. So we agree on one addition, `big_numbers`, and use it for **every** recogniser, before we compare.

## 3. One table for every recogniser

Vosk small is the baseline: the older, small recogniser. The next cell scores every recogniser in both languages, with the peak memory of each run (from the recording runs, Apple M4, 4 threads).

In [ ]:
table = {}
for m, lang in [(m, lang) for m in METHODS for lang in ("en", "fr")] + [("whisper-medium", "fr")]:
        s = score(m, lang)
        run = runs[f"{m}-{lang}"]
        table[(m, lang)] = {**s, "memory_mb": run["peak_memory_mb"]}
        print(f"{m:15} {lang}  WER {s['wer']:5.1f}%  CER {s['cer']:5.1f}%  RTF {s['rtf']:.3f}  memory {run['peak_memory_mb']} MB")

> **Check.** English: Vosk 24.6%, tiny 15.4%, base 11.1%, small 6.7%. French: Vosk 18.8%, tiny 44.4%, base 28.4%, small 13.6%, and medium 7.4% (the last row: run on all French recordings because it looked much better on a first sample). Look at French: the baseline beats two Whisper sizes, and only medium is under 10%.

The next cell draws error against speed. The shaded corner meets both targets: WER at most 10% and RTF at most 0.1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, lang in zip(axes, ("en", "fr")):
    ax.fill_between([0.01, 0.1], 0, 10, color="#EEF3FA")
    for m in METHODS:
        s = table[(m, lang)]
        ax.scatter(s["rtf"], s["wer"], s=60)
        ax.annotate(f"{m} {s['wer']}%", (s["rtf"], s["wer"]), textcoords="offset points", xytext=(6, 3))
    ax.set_xscale("log"); ax.set_xlim(0.01, 0.6); ax.set_ylim(0, 50)
    ax.set_title({"en": "English", "fr": "French"}[lang]); ax.set_xlabel("real-time factor (log)")
axes[0].set_ylabel("WER (%)"); plt.show()

## 4. By speaker gender

> **Predict.** For which recogniser and language do you expect the biggest gap between women and men?

In [ ]:
for m in METHODS:
    print(f"{m:14}", {lang: (score(m, lang, "female")["wer"], score(m, lang, "male")["wer"]) for lang in ("en", "fr")})

A gap of 1 or 2 points can be noise. Because every sentence was read by both, we can compare **inside each sentence**: who got more errors on the same words?

> **Your turn.** Complete `paired_counts(method, language)`. For each sentence, count the word errors of the female and the male recording with `errors(ref, hyp, language)`, and add 1 to `female_worse`, `male_worse` or `ties`.

In [ ]:
def paired_counts(method, language):
    """For every sentence: does the woman's or the man's recording get more word errors?
    Returns {"female_worse": ..., "male_worse": ..., "ties": ...}."""
    rec = transcripts(method, language)
    out = {"female_worse": 0, "male_worse": 0, "ties": 0}
    sentences = sorted({r["sentence_id"] for r in recordings.values() if r["language"] == language})
    for sid in sentences:
        f, m = f"{language}-{sid}-f", f"{language}-{sid}-m"
        ef = errors(recordings[f]["text"], rec[f]["text"], language)[0]
        em = errors(recordings[m]["text"], rec[m]["text"], language)[0]
        out["female_worse" if ef > em else "male_worse" if em > ef else "ties"] += 1
    return out

paired = {(m, lang): paired_counts(m, lang) for m in METHODS for lang in ("en", "fr")}
print(paired[("vosk-small", "fr")])

In [ ]:
expect_hash('paired', paired, 'cb144b16f7e38c88')

> **Check.** For Vosk in French, the female recording had more errors in 62 sentences, the male one in 36, and 17 were ties.

The next cell gives each gap a 95% interval: it resamples the sentences 2,000 times. An interval that includes 0 means "we cannot tell".

In [ ]:
def gap_interval(method, language, reps=2000, seed=41):
    """The gender gap (female WER - male WER, in points) and a 95% interval from resampling the sentences."""
    rec = transcripts(method, language)
    per = []
    for sid in sorted({r["sentence_id"] for r in recordings.values() if r["language"] == language}):
        f, m = f"{language}-{sid}-f", f"{language}-{sid}-m"
        per.append((errors(recordings[f]["text"], rec[f]["text"], language),
                    errors(recordings[m]["text"], rec[m]["text"], language)))
    def gap(items):
        return 100 * (sum(a[0] for a, _ in items) / sum(a[1] for a, _ in items)
                      - sum(b[0] for _, b in items) / sum(b[1] for _, b in items))
    rng = random.Random(seed)
    boots = sorted(gap([rng.choice(per) for _ in per]) for _ in range(reps))
    return round(gap(per), 1), (round(boots[int(0.025 * reps)], 1), round(boots[int(0.975 * reps) - 1], 1))

for m in METHODS:
    print(f"{m:14}", {lang: gap_interval(m, lang) for lang in ("en", "fr")})

## 5. The worst recordings

An average hides the recordings that fail. The next cell prints the three French recordings with the highest WER for the chosen recogniser. Look at what kind of words go wrong.

In [ ]:
m, lang = "whisper-medium", "fr"
rec = transcripts(m, lang)
worst = sorted(((errors(r["text"], rec[u]["text"], lang), u) for u, r in recordings.items()
                if r["language"] == lang), key=lambda x: -x[0][0] / x[0][1])[:3]
for (e, n), u in worst:
    print(f"{u}: {e} errors in {n} words")
    print("  said:  ", recordings[u]["text"])
    print("  wrote: ", rec[u]["text"])

> **Check.** Most errors are names of places and people, and long rare words. A person should check names and numbers in a note before it is saved.

## 6. Can Whisper find the language by itself?

The team also recorded Whisper small without telling it the language. The next cell counts the recordings where it chose the wrong language.

In [ ]:
auto = {**transcripts("whisper-small-auto", "en"), **transcripts("whisper-small-auto", "fr")}
wrong = [u for u, r in auto.items() if r["language_detected"] != recordings[u]["language"]]
print(len(auto), "recordings,", len(wrong), "with the wrong language:", wrong)

> **Check.** 0 wrong. On these clean read sentences detection is reliable; the app can still keep the technician's language setting, because a note can start with a name or a word from the other language.

## 7. Cost per audio hour

> **Your turn.** The server costs US$0.17 an hour (an assumption from the course's cost file). One hour of audio needs `rtf` hours of compute. Fill `cost_per_audio_hour` and `cost_per_month` for every recogniser and language in `table`. The month has 183.3 audio hours, {'en': 0.6, 'fr': 0.4} by language.

In [ ]:
SERVER_USD_PER_HOUR = 0.17          # a CPU server, 4 vCPUs and 16 GB, on demand (the course's assumption)
AUDIO_HOURS_PER_MONTH = 183.3        # the story's volume (an assumption)
cost_per_audio_hour = {}               # (method, language) -> US$ for one hour of audio
cost_per_month = {}                    # (method, language) -> US$ for a month of notes in that language
for (m, lang), s in table.items():
    cost_per_audio_hour[(m, lang)] = round(s["rtf"] * SERVER_USD_PER_HOUR, 4)
    cost_per_month[(m, lang)] = round(s["rtf"] * SERVER_USD_PER_HOUR * AUDIO_HOURS_PER_MONTH * {'en': 0.6, 'fr': 0.4}[lang], 2)
for key in [("whisper-small", "en"), ("whisper-medium", "fr")]:
    print(key, "US$ per audio hour:", cost_per_audio_hour.get(key), "| US$ per month:", cost_per_month.get(key))

In [ ]:
expect_hash('cost_per_audio_hour', cost_per_audio_hour, '237601d931860f45')

In [ ]:
expect_hash('cost_per_month', cost_per_month, '6066d156909bec77')

> **Check.** The chosen recognisers cost about US$0.0228 (English) and US$0.0576 (French) per audio hour on the CPU server: US$2.51 and US$4.23 a month. Compute is not the expensive part; a wrong word that a technician must fix is.

## 8. End to end: the decision on new recordings

The 20 recordings below come from the FLEURS validation split: other sentences, and nobody looked at them while choosing. The next cell loads Whisper small (the English choice) and transcribes all 20 on **your** computer. On a laptop this takes under a minute; on a free Colab CPU, about 2 minutes, plus the model download. For French, it also scores the recorded transcripts of Whisper medium (the French choice) on the same recordings: medium needs about 1.5 GB more to download and is slow on a free CPU.

In [ ]:
import time
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download

SIZE = "small"           # the English choice. With 2 GB free and time, try "medium" (the French choice)
REPOS = {"tiny": ["Systran/faster-whisper-tiny", "d90ca5fe260221311c53c58e660288d3deb8d356"], "small": ["Systran/faster-whisper-small", "536b0662742c02347bc0e980a01041f333bce120"], "base": ["Systran/faster-whisper-base", "ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66"]}
model = WhisperModel(snapshot_download(REPOS[SIZE][0], revision=REPOS[SIZE][1]),
                     device="cpu", compute_type="int8", cpu_threads=4)

def transcribe_note(path, language):
    """The decision in code: one recording in, its text out, with the compute time."""
    x, sr = sf.read(path, dtype="float32")
    start = time.perf_counter()
    segments, _ = model.transcribe(x, language=language, beam_size=1, condition_on_previous_text=False)
    return " ".join(s.text.strip() for s in segments), time.perf_counter() - start

live = {}
for u, r in new_recordings.items():
    text, seconds = transcribe_note(f"data/audio/{u}.flac", r["language"])
    live[u] = {"text": text, "seconds": seconds}
for lang in ("en", "fr"):
    ids = [u for u in live if new_recordings[u]["language"] == lang]
    e = sum(errors(new_recordings[u]["text"], live[u]["text"], lang)[0] for u in ids)
    n = sum(errors(new_recordings[u]["text"], live[u]["text"], lang)[1] for u in ids)
    rtf = sum(live[u]["seconds"] for u in ids) / sum(new_recordings[u]["seconds"] for u in ids)
    print(f"{lang}: {len(ids)} new recordings, WER {100 * e / n:.1f}%, your real-time factor {rtf:.3f}")
medium = transcripts("whisper-medium", "fr", new=True)   # the French choice, recorded on the same 10 recordings
e = sum(errors(new_recordings[u]["text"], medium[u]["text"], "fr")[0] for u in medium)
n = sum(errors(new_recordings[u]["text"], medium[u]["text"], "fr")[1] for u in medium)
print(f"fr: Whisper medium (recorded), WER {100 * e / n:.1f}%")
u = sorted(u for u in live if u.startswith("fr"))[0]
print(u, "said:  ", new_recordings[u]["text"])
print(u, SIZE + ": ", live[u]["text"])
print(u, "medium:", medium[u]["text"])

> **Check.** On the author's laptop: English with small 4.8%, French with small about 12% live (10.7% recorded from the original 32-bit files; these clips are 16-bit, so a word or two can change), French with medium 9.3% (recorded). Compare each WER with the sample's (section 3). With only 10 recordings per language, a few points of difference are normal. Your real-time factor depends on your computer.

> **Change one thing.** Set `SIZE = "tiny"` and run the cell again. Predict first: how much worse is French? If you have the time and 2 GB of free memory, try `SIZE = "medium"` too.

## Next

You chose a recogniser for each language with evidence: one guide, one table, the groups and the worst cases, the cost, and a run on new input. The [lesson page](https://learning.nextia-ai.com/courses/speech/m10/choose-a-recogniser-for-two-languages/) has the full story, the charts and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Meeting transcripts with speakers](https://learning.nextia-ai.com/courses/speech/m10/meeting-transcripts-with-speakers/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m10/choose-a-recogniser-for-two-languages/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The speech in this notebook is real: volunteers read these sentences for FLEURS (Google, CC BY 4.0). Use it with its attribution, and never to identify a speaker.